In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Step 1: Load and Inspect Dataset ---
df = pd.read_csv("train.csv")

# Selected features: Area (GrLivArea), Rooms (TotRmsAbvGrd), Location (Neighborhood)
features = ["GrLivArea", "TotRmsAbvGrd", "Neighborhood"]
target = "SalePrice"

data = df[features + [target]].dropna().copy()

# --- Step 2: Outlier Removal (Lab 1 Preprocessing - IQR Method) ---
for col in ["GrLivArea", "SalePrice"]:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    data = data[(data[col] >= lower_limit) & (data[col] <= upper_limit)]

# --- Step 3: Target Discretization (Low, Medium, High) ---
data["PriceCategory"] = pd.qcut(
    data["SalePrice"], q=3, labels=["Low", "Medium", "High"]
)

# --- Step 4: Categorical Encoding (Lab 1 - One-Hot Encoding) ---
X = pd.get_dummies(data[features], columns=["Neighborhood"], drop_first=True)
y_continuous = data["SalePrice"]
y_categorical = data["PriceCategory"]

# --- Step 5: Train / Test Split ---
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_continuous, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42
)

# --- Step 6: Feature Scaling (Lab 1 - StandardScaler) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Step 7: Linear Regression for Continuous Price (Lab 2) ---
reg_model = LinearRegression()
reg_model.fit(X_train_scaled, y_reg_train)
y_reg_pred = reg_model.predict(X_test_scaled)

# Regression Evaluation Metrics
r2 = r2_score(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)

print("=" * 45)
print("  MODEL 1: LINEAR REGRESSION (PRICE)")
print("=" * 45)
print(f"R² Score : {r2:.4f}")
print(f"MSE      : {mse:,.2f}")
print(f"RMSE     : ${rmse:,.2f}")

# --- Step 8: Random Forest for Category Classification (Lab 5) ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_clf_train)
y_clf_pred = rf_model.predict(X_test)

# Classification Evaluation Metrics
acc = accuracy_score(y_clf_test, y_clf_pred)

print("\n" + "=" * 45)
print("  MODEL 2: RANDOM FOREST (CATEGORY)")
print("=" * 45)
print(f"Accuracy : {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_clf_test, y_clf_pred))

  MODEL 1: LINEAR REGRESSION (PRICE)
R² Score : 0.7244
MSE      : 853,739,550.88
RMSE     : $29,218.82

  MODEL 2: RANDOM FOREST (CATEGORY)
Accuracy : 66.91%

Classification Report:
              precision    recall  f1-score   support

        High       0.76      0.75      0.76        87
         Low       0.74      0.73      0.74        97
      Medium       0.51      0.53      0.52        91

    accuracy                           0.67       275
   macro avg       0.67      0.67      0.67       275
weighted avg       0.67      0.67      0.67       275

